# Hybrid CNN + ViT Road Damage Training (Colab T4)

This notebook trains the project hybrid detector:
- Pretrained **ResNet18/34** CNN branch (local texture)
- Pretrained **DeiT-Tiny/Small** ViT branch (global context)
- From-scratch fusion + FCOS-style head

The same trained model serves both photo inference and video frame inference.

## Runtime + Resource Targets

Recommended for free Colab T4:
- Image size: **640** (or 512 if memory pressure)
- Batch size: **8-16**
- Mixed precision: **fp16**
- Two-stage training: freeze backbones, then partially unfreeze

In [ ]:
!pip -q install timm==1.0.9 torchmetrics==1.4.0 opencv-python-headless==4.10.0.84

In [ ]:
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection.mean_ap import MeanAveragePrecision

from google.colab import drive
drive.mount('/content/drive')

## Copy Repo Into Colab Workspace

Option A: `git clone` your repo.

Option B: upload a zip and extract to `/content/Project 2`.

Then ensure `ai-service/model/hybrid_detector.py` and `ai-service/model/losses.py` are present.

In [ ]:
PROJECT_ROOT = Path('/content/Project 2')
AI_SERVICE_ROOT = PROJECT_ROOT / 'ai-service'
DATA_ROOT = PROJECT_ROOT / 'data' / 'processed' / 'rdd2022'
ARTIFACTS_DIR = PROJECT_ROOT / 'training' / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

import sys
if str(AI_SERVICE_ROOT) not in sys.path:
    sys.path.append(str(AI_SERVICE_ROOT))

from model.hybrid_detector import HybridDetectorConfig, build_hybrid_model
from model.losses import FCOSLoss
from model.postprocess import decode_detections

## (Optional) Run Preprocessing in Colab

If `data/processed/rdd2022` is not ready yet, run the preprocessor after placing raw RDD files.

In [ ]:
# Example only; uncomment after placing raw RDD files in your project path.
# %cd /content/Project 2/ai-service/data
# !python preprocess_rdd2022.py --raw-dir '/content/Project 2/data/raw/RDD2022' --output-dir '/content/Project 2/data/processed/rdd2022' --image-size 640 --min-ravelling-samples 300

In [ ]:
@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 8
    epochs_stage1: int = 2
    epochs_stage2: int = 8
    learning_rate_stage1: float = 3e-4
    learning_rate_stage2: float = 1e-4
    weight_decay: float = 1e-4
    num_workers: int = 2
    score_threshold_eval: float = 0.25
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed: int = 42

cfg = TrainConfig()
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
random.seed(cfg.seed)

print(cfg)

In [ ]:
class CocoDamageDataset(Dataset):
    def __init__(self, images_dir: Path, annotations_file: Path, train: bool):
        self.images_dir = images_dir
        self.train = train

        data = json.loads(annotations_file.read_text(encoding='utf-8'))
        self.images = data['images']
        self.categories = data['categories']

        by_image = {}
        for ann in data['annotations']:
            by_image.setdefault(ann['image_id'], []).append(ann)
        self.by_image = by_image

    def __len__(self):
        return len(self.images)

    def _augment(self, image: np.ndarray, boxes: torch.Tensor):
        if not self.train:
            return image, boxes

        if random.random() < 0.5:
            image = image[:, ::-1, :].copy()
            width = image.shape[1]
            x1 = boxes[:, 0].clone()
            x2 = boxes[:, 2].clone()
            boxes[:, 0] = width - x2
            boxes[:, 2] = width - x1

        if random.random() < 0.2:
            gain = random.uniform(0.85, 1.15)
            image = np.clip(image.astype(np.float32) * gain, 0, 255).astype(np.uint8)

        return image, boxes

    def __getitem__(self, idx):
        item = self.images[idx]
        image = cv2.imread(str(self.images_dir / item['file_name']))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        anns = self.by_image.get(item['id'], [])
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann['bbox']
            boxes.append([x, y, x + w, y + h])
            labels.append(ann['category_id'])

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.float32)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.float32)

        image, boxes = self._augment(image, boxes)

        image = np.ascontiguousarray(image)
        image_tensor = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0

        if boxes.shape[0] == 0:
            target = torch.zeros((0, 5), dtype=torch.float32)
        else:
            target = torch.cat([boxes, labels.unsqueeze(1)], dim=1)

        return image_tensor, target


def collate_fn(batch):
    images = torch.stack([x[0] for x in batch], dim=0)
    targets = [x[1] for x in batch]
    return images, targets

In [ ]:
train_ds = CocoDamageDataset(
    images_dir=DATA_ROOT / 'images' / 'train',
    annotations_file=DATA_ROOT / 'annotations_train.json',
    train=True,
)
val_ds = CocoDamageDataset(
    images_dir=DATA_ROOT / 'images' / 'val',
    annotations_file=DATA_ROOT / 'annotations_val.json',
    train=False,
)

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True,
    collate_fn=collate_fn,
)

print('Train images:', len(train_ds), 'Val images:', len(val_ds))

In [ ]:
model_cfg = HybridDetectorConfig(
    num_classes=4,
    class_names=('pothole', 'linear_crack', 'alligator_crack', 'edge_break'),
    cnn_variant='resnet34',
    cnn_output_stage='layer3',
    vit_variant='deit_small_patch16_224',
    pretrained_backbones=True,
)

model = build_hybrid_model(model_cfg).to(cfg.device)
criterion = FCOSLoss(num_classes=model_cfg.num_classes, stride=model.stride)
scaler = torch.amp.GradScaler(enabled=(cfg.device == 'cuda'))

def build_optimizer(lr: float):
    trainable = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=cfg.weight_decay)

In [ ]:
def to_metric_format(decoded, targets):
    preds = []
    refs = []

    for pred_det, tgt in zip(decoded, targets):
        if len(pred_det) == 0:
            preds.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'scores': torch.zeros((0,), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            boxes = torch.tensor([d['bbox'] for d in pred_det], dtype=torch.float32)
            scores = torch.tensor([d['confidence'] for d in pred_det], dtype=torch.float32)
            labels = torch.tensor([d['class_id'] for d in pred_det], dtype=torch.int64)
            preds.append({'boxes': boxes, 'scores': scores, 'labels': labels})

        if tgt.shape[0] == 0:
            refs.append({
                'boxes': torch.zeros((0, 4), dtype=torch.float32),
                'labels': torch.zeros((0,), dtype=torch.int64),
            })
        else:
            refs.append({
                'boxes': tgt[:, :4].cpu().float(),
                'labels': tgt[:, 4].cpu().long(),
            })

    return preds, refs


def iou_xyxy(a: torch.Tensor, b: torch.Tensor) -> float:
    x1 = max(float(a[0]), float(b[0]))
    y1 = max(float(a[1]), float(b[1]))
    x2 = min(float(a[2]), float(b[2]))
    y2 = min(float(a[3]), float(b[3]))
    inter = max(0.0, x2 - x1) * max(0.0, y2 - y1)
    area_a = max(0.0, float(a[2] - a[0])) * max(0.0, float(a[3] - a[1]))
    area_b = max(0.0, float(b[2] - b[0])) * max(0.0, float(b[3] - b[1]))
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def batch_iou_f1(decoded, targets, iou_thresh=0.5):
    tp = 0
    fp = 0
    fn = 0
    iou_sum = 0.0
    match_count = 0

    for pred_det, tgt in zip(decoded, targets):
        gt = tgt.cpu()
        used_gt = set()

        for pred in pred_det:
            pbox = torch.tensor(pred['bbox'], dtype=torch.float32)
            plabel = int(pred['class_id'])

            best_iou = 0.0
            best_idx = -1
            for gi in range(gt.shape[0]):
                if gi in used_gt:
                    continue
                if int(gt[gi, 4].item()) != plabel:
                    continue
                iou = iou_xyxy(pbox, gt[gi, :4])
                if iou > best_iou:
                    best_iou = iou
                    best_idx = gi

            if best_idx >= 0 and best_iou >= iou_thresh:
                tp += 1
                used_gt.add(best_idx)
                iou_sum += best_iou
                match_count += 1
            else:
                fp += 1

        fn += max(0, gt.shape[0] - len(used_gt))

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)
    mean_iou = iou_sum / (match_count + 1e-8)
    return float(mean_iou), float(f1)


def run_eval(model, loader):
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    iou_vals = []
    f1_vals = []

    model.eval()
    with torch.no_grad():
        for images, targets in loader:
            images = images.to(cfg.device, non_blocking=True)
            outputs = model(images)

            decoded = decode_detections(
                cls_logits=outputs['cls_logits'],
                bbox_reg=outputs['bbox_reg'],
                centerness=outputs['centerness'],
                stride=model.stride,
                class_names=model_cfg.class_names,
                score_threshold=cfg.score_threshold_eval,
                nms_iou_threshold=0.5,
                top_k=200,
                image_hw=(cfg.image_size, cfg.image_size),
            )

            preds, refs = to_metric_format(decoded, targets)
            metric.update(preds, refs)

            mean_iou, f1 = batch_iou_f1(decoded, targets, iou_thresh=0.5)
            iou_vals.append(mean_iou)
            f1_vals.append(f1)

    map_scores = metric.compute()
    return {
        'mAP': float(map_scores.get('map', torch.tensor(0.0)).item()),
        'mAP@50': float(map_scores.get('map_50', torch.tensor(0.0)).item()),
        'mean_iou': float(np.mean(iou_vals) if iou_vals else 0.0),
        'f1': float(np.mean(f1_vals) if f1_vals else 0.0),
    }

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running = 0.0

    for images, targets in loader:
        images = images.to(cfg.device, non_blocking=True)
        targets = [t.to(cfg.device) for t in targets]

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', enabled=(cfg.device == 'cuda'), dtype=torch.float16):
            outputs = model(images)
            losses = criterion(outputs['cls_logits'], outputs['bbox_reg'], outputs['centerness'], targets)
            loss = losses['loss_total']

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += float(loss.item())

    return running / max(1, len(loader))


history = []
best_map50 = -1.0
best_ckpt = ARTIFACTS_DIR / 'hybrid_detector_best.pt'

# Stage 1: freeze pretrained backbones, train only fusion + head.
model.freeze_backbones()
optimizer = build_optimizer(cfg.learning_rate_stage1)

for epoch in range(cfg.epochs_stage1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    metrics = run_eval(model, val_loader)
    row = {'stage': 1, 'epoch': epoch + 1, 'train_loss': train_loss, **metrics}
    history.append(row)
    print(row)

    if metrics['mAP@50'] > best_map50:
        best_map50 = metrics['mAP@50']
        torch.save({'model_state': model.state_dict(), 'config': model_cfg.__dict__, 'metrics': row}, best_ckpt)

# Stage 2: unfreeze selected backbone layers for fine-tuning.
model.unfreeze_vit_last_blocks(num_blocks=2)
model.unfreeze_cnn_last_stage()
optimizer = build_optimizer(cfg.learning_rate_stage2)

for epoch in range(cfg.epochs_stage2):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    metrics = run_eval(model, val_loader)
    row = {'stage': 2, 'epoch': epoch + 1, 'train_loss': train_loss, **metrics}
    history.append(row)
    print(row)

    if metrics['mAP@50'] > best_map50:
        best_map50 = metrics['mAP@50']
        torch.save({'model_state': model.state_dict(), 'config': model_cfg.__dict__, 'metrics': row}, best_ckpt)

history_path = ARTIFACTS_DIR / 'train_history.json'
history_path.write_text(json.dumps(history, indent=2), encoding='utf-8')
print('Best mAP@50:', best_map50)
print('Best checkpoint:', best_ckpt)

In [ ]:
# Load best weights before ONNX export
state = torch.load(ARTIFACTS_DIR / 'hybrid_detector_best.pt', map_location=cfg.device)
model.load_state_dict(state['model_state'])
model.eval()

class OnnxExportWrapper(nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector

    def forward(self, x):
        out = self.detector(x)
        return out['cls_logits'], out['bbox_reg'], out['centerness']

wrapper = OnnxExportWrapper(model).to(cfg.device)
dummy = torch.randn(1, 3, cfg.image_size, cfg.image_size, device=cfg.device)
onnx_path = ARTIFACTS_DIR / 'hybrid_detector.onnx'

torch.onnx.export(
    wrapper,
    dummy,
    str(onnx_path),
    input_names=['image'],
    output_names=['cls_logits', 'bbox_reg', 'centerness'],
    dynamic_axes={
        'image': {0: 'batch'},
        'cls_logits': {0: 'batch'},
        'bbox_reg': {0: 'batch'},
        'centerness': {0: 'batch'},
    },
    opset_version=17,
)

print('ONNX exported:', onnx_path)

In [ ]:
# Optional: copy final artifacts to Drive for download
DRIVE_OUT = Path('/content/drive/MyDrive/road-damage-artifacts')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

for src in [
    ARTIFACTS_DIR / 'hybrid_detector_best.pt',
    ARTIFACTS_DIR / 'hybrid_detector.onnx',
    ARTIFACTS_DIR / 'train_history.json',
]:
    if src.exists():
        dst = DRIVE_OUT / src.name
        dst.write_bytes(src.read_bytes())
        print('Saved:', dst)

## Expected Outputs

After successful run, confirm these files exist:
- `training/artifacts/hybrid_detector_best.pt`
- `training/artifacts/hybrid_detector.onnx`
- `training/artifacts/train_history.json`

Copy the weight and ONNX files back into this repo for serving:
- `ai-service/weights/hybrid_detector_best.pt`
- `ai-service/weights/hybrid_detector.onnx`